In [1]:
import json
import requests

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            if len(block['buttons']) != 1:
                unmodified_blocks.append({
                    "block": block,
                    "reason": "Parent block does not contain exactly one button"
                })
                updated_blocks.append(block)
                continue

            button = block['buttons'][0]
            if not ('blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE):
                unmodified_blocks.append({
                    "block": block,
                    "reason": "Button does not contain exactly one launch block"
                })
                updated_blocks.append(block)
                continue

            launch_block = button['blocks'][0]
            if not button.get('enabled', True):
                unmodified_blocks.append({
                    "block": block,
                    "reason": "Button is not enabled"
                })
                updated_blocks.append(block)
                continue

            if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                unmodified_blocks.append({
                    "block": block,
                    "reason": "Button contains unexpected properties"
                })
                updated_blocks.append(block)
                continue

            if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                unmodified_blocks.append({
                    "block": block,
                    "reason": "Launch block contains unexpected properties"
                })
                updated_blocks.append(block)
                continue

            stats['total_launch_blocks_before'] += 1
            new_block = transform_launch_block(button, launch_block)
            updated_blocks.append(new_block)
        elif 'blocks' in block:
            block['blocks'] = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks

def process_flows(flows):
    """Process each flow and store in fixed_flows."""
    fixed_flows = []
    for flow in flows:
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'] = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 52
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 40

Unmodified Launch Blocks:
Reason: Parent block does not contain exactly one button
{
  "type": "actions",
  "buttons": [
    {
      "label": "Add Gig",
      "color": "primary",
      "blocks": [
        {
          "type": "launch",
          "adapter": "backstage",
          "workflowId": "addgig"
        }
      ],
      "enabled": true
    },
    {
      "label": "Manage Venues",
      "color": "primary",
      "blocks": [
        {
          "type": "launch",
          "adapter": "backstage",
          "workflowId": "managevenues"
        }
      ],
      "enabled": true
    },
    {
      "label": "Manage Artists",
      "color": "primary",
      "blocks": [
        {
          "type": "launch",
          "adapter": "backstage",
          "workflowId": "manageartists"
        }
      ],
      "enabled": true
    }
  ],
  "blockComment

In [2]:
import json
import requests

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_blocks = []
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_blocks.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_blocks

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons = process_buttons(block['buttons'])
            updated_blocks.extend(transformed_buttons)
        elif 'blocks' in block:
            block['blocks'] = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks

def process_flows(flows):
    """Process each flow and store in fixed_flows."""
    fixed_flows = []
    for flow in flows:
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'] = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [3]:
import json
import requests

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_blocks = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_blocks.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_blocks, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
            updated_blocks.extend(transformed_buttons)
        elif 'blocks' in block:
            block['blocks'] = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows."""
    fixed_flows = []
    for flow in flows:
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            updated_blocks, has_launch_block = process_blocks(flow['blocks'])
            flow['blocks'] = updated_blocks
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            if has_launch_block:
                unfixed_flows.append(flow)
        fixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [4]:
import json
import requests

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks and unfixed flows
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_blocks = []
    contains_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            contains_launch_block = True
            launch_block = button['blocks'][0]

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_blocks.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_blocks, contains_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    flow_contains_launch_block = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, contains_launch_block = process_buttons(block['buttons'])
            flow_contains_launch_block |= contains_launch_block
            updated_blocks.extend(transformed_buttons)
        elif 'blocks' in block:
            result, contains_launch_block = process_blocks(block['blocks'])
            flow_contains_launch_block |= contains_launch_block
            block['blocks'] = result
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)

    return updated_blocks, flow_contains_launch_block

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows if applicable."""
    fixed_flows = []
    for flow in flows:
        if 'blocks' in flow:
            transformed_blocks, contains_launch_block = process_blocks(flow['blocks'])
            flow['blocks'] = transformed_blocks
            if contains_launch_block:
                unfixed_flows.append(flow)
            fixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 104
Total 'launch' blocks after transformation: 0
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
   

In [5]:
import json
import requests

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_blocks = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_blocks.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_blocks, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
            updated_blocks.extend(transformed_buttons)
        elif 'blocks' in block:
            block['blocks'] = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows."""
    fixed_flows = []
    for flow in flows:
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            updated_blocks, has_launch_block = process_blocks(flow['blocks'])
            flow['blocks'] = updated_blocks
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            if has_launch_block:
                unfixed_flows.append(flow)
        fixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [6]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons, parent_flow, original_flow):
    """Process and transform buttons."""
    transformed_blocks = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_blocks.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_blocks, has_launch_block

def process_blocks(blocks, parent_flow, original_flow):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'], parent_flow, original_flow)
            if has_launch_block:
                block_contains_launch = True
                unfixed_copy = copy.deepcopy(original_flow)
                unfixed_flows.append(unfixed_copy)
            updated_blocks.extend(transformed_buttons)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'], parent_flow, original_flow)
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'], flow, original_flow)
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [7]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
fixed_flows = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
            updated_blocks.extend(transformed_buttons)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows."""
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            if has_launch_block:
                unfixed_flows.append(original_flow)
        fixed_flows.append(flow)

# Process the flows
process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [8]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
        "adapterName": get_value(launch_block, 'adapter'),
        "workflowId": get_value(launch_block, 'workflowId'),
        "adapterNameGetter": get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'], ""),
        "workflowIdGetter": get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'], "")
    }
    stats['total_link_action_blocks'] += 1
    return new_block

def process_flow(flow):
    """Process and potentially transform a flow containing actions with launch blocks."""
    original_flow = copy.deepcopy(flow)
    transformed = False

    def process_blocks(blocks):
        nonlocal transformed
        updated_blocks = []
        for block in blocks:
            if block['type'] == PARENT_BLOCK_TYPE:
                for button in block.get('buttons', []):
                    if 'blocks' in button:
                        for subblock in button['blocks']:
                            if subblock['type'] == LAUNCH_BLOCK_TYPE:
                                stats['total_launch_blocks_before'] += 1
                                new_block = transform_launch_block(button, subblock)
                                updated_blocks.append(new_block)
                                transformed = True
                                continue
            if 'blocks' in block:
                block['blocks'] = process_blocks(block['blocks'])
            updated_blocks.append(block)
        return updated_blocks

    flow['blocks'] = process_blocks(flow.get('blocks', []))
    return original_flow, flow if transformed else original_flow

fixed_flows = []
unfixed_flows = []

# Process each flow
for flow in flows_data:
    original_flow, transformed_flow = process_flow(flow)
    if original_flow and transformed_flow:
        unfixed_flows.append(original_flow)
        fixed_flows.append(transformed_flow)

# Save the flows to files for comparison
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=4)

with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=4)

# Print statistics
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")


Statistics:
Total 'launch' blocks before transformation: 131
Total 'link-action' blocks created: 131


In [9]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_blocks = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_blocks.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_blocks, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
            updated_blocks.extend(transformed_buttons)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [10]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons, parent_flow, original_flow):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'], None, None)
            if has_launch_block:
                block_contains_launch = True
            updated_blocks.append({
                "type": PARENT_BLOCK_TYPE,
                "buttons": transformed_buttons
            })
        elif 'blocks' in block:
            child_blocks, child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append({**block, "blocks": child_blocks})
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [11]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_blocks = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_blocks.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })

    return transformed_blocks, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [12]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


KeyError: 'type'

In [13]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0]['type'] == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block['type'] == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block['type'] == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


KeyError: 'type'

In [14]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0].get('type') == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [15]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1:
            launch_block = button['blocks'][0]
            if launch_block.get('type') == LAUNCH_BLOCK_TYPE:
                has_launch_block = True
                if not button.get('enabled', True):
                    reason = "Button is not enabled"
                elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                    reason = "Button contains unexpected properties"
                elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                    reason = "Launch block contains unexpected properties"
                else:
                    stats['total_launch_blocks_before'] += 1
                    new_block = transform_launch_block(button, launch_block)
                    transformed_buttons.append(new_block)
                    continue  # Skip appending the original button
                unmodified_blocks.append({
                    "block": button,
                    "reason": reason
                })
        else:
            reason = "Button does not contain exactly one launch block"
            unmodified_blocks.append({
                "block": button,
                "reason": reason
            })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workflowId": "getArtists"
    },
    {
      "type": "context-save",
      "key": "state.local.artistList"
    },
    {
      "

In [16]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1:
            launch_block = button['blocks'][0]
            if launch_block.get('type') == LAUNCH_BLOCK_TYPE:
                if not button.get('enabled', True):
                    reason = "Button is not enabled"
                elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                    reason = "Button contains unexpected properties"
                elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                    reason = "Launch block contains unexpected properties"
                else:
                    stats['total_launch_blocks_before'] += 1
                    new_block = transform_launch_block(button, launch_block)
                    transformed_buttons.append(new_block)
                    continue  # Skip appending the original button
            else:
                reason = "Button does not contain a launch block"
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain a launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workf

In [17]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment', 'enabled'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0].get('type') == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, ALLOWED_IGNORE_KEYS):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 12
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 0

Unmodified Launch Blocks:
Reason: Button contains unexpected properties
{
  "label": "Dashboard",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "dashboard"
    }
  ],
  "enabled": true
}
Reason: Button contains unexpected properties
{
  "label": "Dashboard",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "dashboard"
    }
  ],
  "enabled": true
}
Reason: Button contains unexpected properties
{
  "label": "Back",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "dashboard"
    }
  ],
  "enabled": true
}
Reason: Button contains unexpected properties
{
  "label": "Add Gig",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
    

In [18]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0].get('type') == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                if len(transformed_buttons) != len(block['buttons']):
                    updated_blocks.extend(transformed_buttons)
                else:
                    updated_blocks.append(block)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        if has_launch_block:
            unfixed_flows.append(original_flow)
        else:
            unfixed_flows.append(flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [19]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1 and button['blocks'][0].get('type') == LAUNCH_BLOCK_TYPE:
            launch_block = button['blocks'][0]
            has_launch_block = True

            if not button.get('enabled', True):
                reason = "Button is not enabled"
            elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                reason = "Button contains unexpected properties"
            elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                reason = "Launch block contains unexpected properties"
            else:
                stats['total_launch_blocks_before'] += 1
                new_block = transform_launch_block(button, launch_block)
                transformed_buttons.append(new_block)
                continue  # Skip appending the original button
        else:
            reason = "Button does not contain exactly one launch block"

        unmodified_blocks.append({
            "block": button,
            "reason": reason
        })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                if len(transformed_buttons) != len(block['buttons']):
                    updated_blocks.extend(transformed_buttons)
                else:
                    updated_blocks.append(block)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Button does not contain exactly one launch block
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
  

In [20]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1:
            launch_block = button['blocks'][0]
            if launch_block.get('type') == LAUNCH_BLOCK_TYPE:
                has_launch_block = True
                if not button.get('enabled', True):
                    reason = "Button is not enabled"
                elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                    reason = "Button contains unexpected properties"
                elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                    reason = "Launch block contains unexpected properties"
                else:
                    stats['total_launch_blocks_before'] += 1
                    new_block = transform_launch_block(button, launch_block)
                    transformed_buttons.append(new_block)
                    continue  # Skip appending the original button
                unmodified_blocks.append({
                    "block": button,
                    "reason": reason
                })
        else:
            reason = "Button does not contain exactly one launch block"
            unmodified_blocks.append({
                "block": button,
                "reason": reason
            })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workflowId": "getArtists"
    },
    {
      "type": "context-save",
      "key": "state.local.artistList"
    },
    {
      "

In [22]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1:
            launch_block = button['blocks'][0]
            if launch_block.get('type') == LAUNCH_BLOCK_TYPE:
                has_launch_block = True
                if not button.get('enabled', True):
                    reason = "Button is not enabled"
                elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                    reason = "Button contains unexpected properties"
                elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                    reason = "Launch block contains unexpected properties"
                else:
                    stats['total_launch_blocks_before'] += 1
                    new_block = transform_launch_block(button, launch_block)
                    transformed_buttons.append(new_block)
                    continue  # Skip appending the original button
                unmodified_blocks.append({
                    "block": button,
                    "reason": reason
                })
        else:
            reason = "Button does not contain exactly one launch block"
            unmodified_blocks.append({
                "block": button,
                "reason": reason
            })
        transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                # If any button was transformed, replace the entire actions block
                updated_blocks.append({
                    "type": PARENT_BLOCK_TYPE,
                    "buttons": transformed_buttons
                })
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workflowId": "getArtists"
    },
    {
      "type": "context-save",
      "key": "state.local.artistList"
    },
    {
      "

In [23]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1:
            launch_block = button['blocks'][0]
            if launch_block.get('type') == LAUNCH_BLOCK_TYPE:
                has_launch_block = True
                if not button.get('enabled', True):
                    reason = "Button is not enabled"
                elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                    reason = "Button contains unexpected properties"
                elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                    reason = "Launch block contains unexpected properties"
                else:
                    stats['total_launch_blocks_before'] += 1
                    new_block = transform_launch_block(button, launch_block)
                    transformed_buttons.append(new_block)
                    continue  # Skip appending the original button
                unmodified_blocks.append({
                    "block": button,
                    "reason": reason
                })
            else:
                transformed_buttons.append(button)
        else:
            reason = "Button does not contain exactly one launch block"
            unmodified_blocks.append({
                "block": button,
                "reason": reason
            })
            transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                if len(transformed_buttons) == len(block['buttons']):  # Ensure all buttons were transformed
                    updated_blocks.extend(transformed_buttons)
                else:  # If not all buttons were transformed, keep the original block
                    updated_blocks.append(block)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workflowId": "getArtists"
    },
    {
      "type": "context-save",
      "key": "state.local.artistList"
    },
    {
      "

In [24]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1:
            launch_block = button['blocks'][0]
            if launch_block.get('type') == LAUNCH_BLOCK_TYPE:
                has_launch_block = True
                if not button.get('enabled', True):
                    reason = "Button is not enabled"
                elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                    reason = "Button contains unexpected properties"
                elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                    reason = "Launch block contains unexpected properties"
                else:
                    stats['total_launch_blocks_before'] += 1
                    new_block = transform_launch_block(button, launch_block)
                    transformed_buttons.append(new_block)
                    continue  # Skip appending the original button
                unmodified_blocks.append({
                    "block": button,
                    "reason": reason
                })
            else:
                transformed_buttons.append(button)
        else:
            reason = "Button does not contain exactly one launch block"
            unmodified_blocks.append({
                "block": button,
                "reason": reason
            })
            transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                if len(transformed_buttons) == len(block['buttons']):  # Ensure all buttons were transformed
                    updated_blocks.extend(transformed_buttons)
                else:  # If not all buttons were transformed, keep the original block
                    updated_blocks.append(block)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workflowId": "getArtists"
    },
    {
      "type": "context-save",
      "key": "state.local.artistList"
    },
    {
      "

In [25]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    has_launch_block = False
    for button in buttons:
        if 'blocks' in button and len(button['blocks']) == 1:
            launch_block = button['blocks'][0]
            if launch_block.get('type') == LAUNCH_BLOCK_TYPE:
                has_launch_block = True
                if not button.get('enabled', True):
                    reason = "Button is not enabled"
                elif has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
                    reason = "Button contains unexpected properties"
                elif has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
                    reason = "Launch block contains unexpected properties"
                else:
                    stats['total_launch_blocks_before'] += 1
                    new_block = transform_launch_block(button, launch_block)
                    transformed_buttons.append(new_block)
                    continue  # Skip appending the original button
                unmodified_blocks.append({
                    "block": button,
                    "reason": reason
                })
            else:
                transformed_buttons.append(button)
        else:
            reason = "Button does not contain exactly one launch block or contains multiple blocks"
            unmodified_blocks.append({
                "block": button,
                "reason": reason
            })
            transformed_buttons.append(button)  # Ensure original button is preserved

    return transformed_buttons, has_launch_block

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    block_contains_launch = False
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            transformed_buttons, has_launch_block = process_buttons(block['buttons'])
            if has_launch_block:
                block_contains_launch = True
                if len(transformed_buttons) == len(block['buttons']):  # Ensure all buttons were transformed
                    updated_blocks.extend(transformed_buttons)
                else:  # If not all buttons were transformed, keep the original block
                    updated_blocks.append(block)
            else:
                updated_blocks.append(block)
        elif 'blocks' in block:
            block['blocks'], child_contains_launch = process_blocks(block['blocks'])
            if child_contains_launch:
                block_contains_launch = True
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    
    return updated_blocks, block_contains_launch

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 116
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 104

Unmodified Launch Blocks:
Reason: Launch block contains unexpected properties
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not contain exactly one launch block or contains multiple blocks
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workflowId": "getArtists"
    },
    {
      "type": "context-save",
      "key": "state.local.arti

In [27]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block['buttons']
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
            else:
                updated_blocks.append(block)
                for button in buttons:
                    unmodified_blocks.append({
                        "block": button,
                        "reason": "Button does not meet transformation criteria"
                    })
        elif 'blocks' in block:
            block['blocks'], _ = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 12
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 95

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
  "color": "default",
  "enabledGetter": "false && !(state.flags.bandsintown.artistExists)",
  "blocks": [
    {
      "type": "message",
      "message": "loading"
    },
    {
      "type": "gosub",
      "adapterName": "bandsintown",
      "workflowId": "getArtists"
    },
    {
      "type": "context-save",
      "key": "state.local.artistList"
    },
    {
      "type": "debug"
    }
  ],
  "enabled": null
}
Reason: Button does not meet transformation criteria
{
  "label": "OK",
  "color": "primary",
  "blocks": [
    {
      "type": "dispatch",
      "action": "resetApp"
    }
  ],
  "enabled": t

In [28]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['endpoint', 'valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'endpoint'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block['buttons']
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
                stats['total_launch_blocks_before'] += len(buttons)
            else:
                updated_blocks.append(block)
                for button in buttons:
                    unmodified_blocks.append({
                        "block": button,
                        "reason": "Button does not meet transformation criteria"
                    })
        elif 'blocks' in block:
            block['blocks'], _ = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 95
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 83

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [29]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block['buttons']
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
                stats['total_launch_blocks_before'] += len(buttons)
            else:
                updated_blocks.append(block)
                for button in buttons:
                    unmodified_blocks.append({
                        "block": button,
                        "reason": "Button does not meet transformation criteria"
                    })
        elif 'blocks' in block:
            block['blocks'], _ = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 95
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 83

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [30]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block['buttons']
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
                stats['total_launch_blocks_before'] += len(buttons)
            else:
                updated_blocks.append(block)
                for button in buttons:
                    unmodified_blocks.append({
                        "block": button,
                        "reason": "Button does not meet transformation criteria"
                    })
        elif 'blocks' in block:
            block['blocks'], _ = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 95
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 83

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [31]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block['buttons']
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
                stats['total_launch_blocks_before'] += len(buttons)
            else:
                updated_blocks.append(block)
                for button in buttons:
                    unmodified_blocks.append({
                        "block": button,
                        "reason": "Button does not meet transformation criteria"
                    })
        elif 'blocks' in block:
            block['blocks'], _ = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 95
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 83

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [32]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block['buttons']
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
                stats['total_launch_blocks_before'] += len(buttons)
            else:
                updated_blocks.append(block)
                for button in buttons:
                    unmodified_blocks.append({
                        "block": button,
                        "reason": "Button does not meet transformation criteria"
                    })
        elif 'blocks' in block:
            block['blocks'], _ = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))

# Verification to check transformation correctness
verification_flow = {
    "type": "actions",
    "buttons": [
        {
            "label": "Launch",
            "blocks": [
                {
                    "type": "launch",
                    "valueGetters": {
                        "adapter": "data.adapterName",
                        "workflowId": "data.id"
                    }
                }
            ]
        }
    ]
}

fixed_verification_flow = {
    "type": "link-action",
    "label": "Launch",
    "adapterNameGetter": "data.adapterName",
    "workflowIdGetter": "data.id"
}

verification_transformed = transform_launch_block(verification_flow['buttons'][0], verification_flow['buttons'][0]['blocks'][0])

assert verification_transformed == fixed_verification_flow, "Verification failed!"
print("Verification passed!")


Statistics:
Total 'launch' blocks before transformation: 95
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 83

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [33]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block['buttons']
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
                stats['total_launch_blocks_before'] += len(buttons)
            else:
                updated_blocks.append(block)
                for button in buttons:
                    unmodified_blocks.append({
                        "block": button,
                        "reason": "Button does not meet transformation criteria"
                    })
        elif 'blocks' in block:
            block['blocks'], _ = process_blocks(block['blocks'])
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))

# Verification to check transformation correctness
verification_flow = {
    "type": "actions",
    "buttons": [
        {
            "label": "Launch",
            "blocks": [
                {
                    "type": "launch",
                    "valueGetters": {
                        "adapter": "data.adapterName",
                        "workflowId": "data.id"
                    }
                }
            ]
        }
    ]
}

fixed_verification_flow = {
    "type": "link-action",
    "label": "Launch",
    "adapterNameGetter": "data.adapterName",
    "workflowIdGetter": "data.id"
}

verification_transformed = transform_launch_block(verification_flow['buttons'][0], verification_flow['buttons'][0]['blocks'][0])

assert verification_transformed == fixed_verification_flow, "Verification failed!"
print("Verification passed!")


Statistics:
Total 'launch' blocks before transformation: 95
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 83

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [34]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'], adapter_name)
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'], workflow_id)

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if block.get('type') == PARENT_BLOCK_TYPE:
            buttons = block.get('buttons', [])
            if can_transform_buttons(buttons):
                transformed_buttons = process_buttons(buttons)
                updated_blocks.extend(transformed_buttons)
                stats['total_launch_blocks_before'] += len(buttons)
            else:
                updated_blocks.append(block)
        elif isinstance(block, dict):
            for key, value in block.items():
                if isinstance(value, list):
                    block[key], _ = process_blocks(value)
            updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


AttributeError: 'str' object has no attribute 'get'

In [35]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'], adapter_name)
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'], workflow_id)

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                # Recursively process nested blocks
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 98
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 86

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [36]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'], adapter_name)
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'], workflow_id)

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))

# Verification to check transformation correctness
verification_flow = {
    "type": "actions",
    "buttons": [
        {
            "label": "Launch",
            "blocks": [
                {
                    "type": "launch",
                    "valueGetters": {
                        "adapter": "data.adapterName",
                        "workflowId": "data.id"
                    }
                }
            ]
        }
    ]
}

fixed_verification_flow = {
    "type": "link-action",
    "label": "Launch",
    "adapterNameGetter": "data.adapterName",
    "workflowIdGetter": "data.id"
}

verification_transformed = transform_launch_block(verification_flow['buttons'][0], verification_flow['buttons'][0]['blocks'][0])

assert verification_transformed == fixed_verification_flow, "Verification failed!"
print("Verification passed!")


Statistics:
Total 'launch' blocks before transformation: 98
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 86

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [37]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 98
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 86

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [38]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_nested_blocks(block):
    """Process nested blocks within a dictionary."""
    for key, value in block.items():
        if isinstance(value, list):
            block[key], _ = process_blocks(value)
        elif isinstance(value, dict):
            process_nested_blocks(value)
    return block

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 98
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 86

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Close",
  "blocks": [
    {
      "type": "init"
    }
  ]
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "editEvent",
      "context": {
        "id": "data.id",
        "artist": "data.headline_artist_id",
        "type": "data.type"
      }
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Back to start page",
  "color": "secondary",
  "blocks": [
    {
      "type": "launch",
      "adapter": "bandsintown",
      "workflowId": "start"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Load managed artists",
 

In [39]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value)
                    elif isinstance(value, dict):
                        block[key] = process_nested_blocks(value)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_nested_blocks(block):
    """Process nested blocks within a dictionary."""
    for key, value in block.items():
        if isinstance(value, list):
            block[key], _ = process_blocks(value)
        elif isinstance(value, dict):
            block[key] = process_nested_blocks(value)
    return block

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))


Statistics:
Total 'launch' blocks before transformation: 100
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 88

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updategig"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updateartist"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "ada

In [40]:
import json

def count_blocks_and_list_flows(file_path, block_type):
    with open(file_path, 'r') as file:
        data = json.load(file)

    def recursive_count(data):
        count = 0
        if isinstance(data, list):
            for item in data:
                count += recursive_count(item)
        elif isinstance(data, dict):
            if data.get('type') == block_type:
                count += 1
            for key, value in data.items():
                count += recursive_count(value)
        return count

    def extract_flow_titles(data):
        titles = []
        if isinstance(data, list):
            for item in data:
                titles.extend(extract_flow_titles(item))
        elif isinstance(data, dict):
            if 'title' in data:
                titles.append(data['title'])
            for key, value in data.items():
                titles.extend(extract_flow_titles(value))
        return titles

    block_count = recursive_count(data)
    flow_titles = extract_flow_titles(data)

    return block_count, flow_titles

# Define file paths and block types
file_paths = ['unfixed_flows.json', 'fixed_flows.json']
block_types = ['launch', 'link-action']

# Count and print occurrences for each block type in each file, and list flow names
for file_path in file_paths:
    print(f"Counts and Flow Titles in {file_path}:")
    for block_type in block_types:
        count, flow_titles = count_blocks_and_list_flows(file_path, block_type)
        print(f"{block_type}: {count}")
    print("Flow Titles:")
    for title in flow_titles:
        print(title)
    print("\n")


Counts and Flow Titles in unfixed_flows.json:
launch: 222
link-action: 1
Flow Titles:
List Documents
Google Drive Client ID
List Comments
List Documents
Files
List Documents
List Documents
Google Drive Token
Dummy Workflow 1
Double Dynadot Domains
Key 1
Key 2
Kendraio Backstage Add Gig
Headline Artist
Venue
Description
Start Date
Start Time
Status
Lineup
Tickets Manager
Kendraio Backstage Add Gig
Headline Artist
Venue
Description
Start Date
Start Time
Status
Lineup
Tickets Manager
Kendraio Backstage
Venue
City
Country
Longitude
Latitude
Kendraio Backstage Dashboard
Hasura Secret
Kendraio Backstage (create)
Hasura Secret
Headline Artist
Description
Start Date
Start Time
Status
Type
Venue ID
Lineup
Tickets Manager
Link
Tickets Manager
Kendraio Backstage Dashboard
Kendraio Backstage Find Artist
Artist
Kendraio Backstage Find Artist
Kendraio Backstage Find Venue
Venue
Kendraio Backstage Manage Artists
Artist Name
Kendraio Backstage Manage Venues
Name
City
Country
Latitude
Longitude
Kendrai

In [41]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []
launch_block_urls = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value)
                    elif isinstance(value, dict):
                        block[key] = process_nested_blocks(value)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_nested_blocks(block):
    """Process nested blocks within a dictionary."""
    for key, value in block.items():
        if isinstance(value, list):
            block[key], _ = process_blocks(value)
        elif isinstance(value, dict):
            block[key] = process_nested_blocks(value)
    return block

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], has_launch_block = process_blocks(flow['blocks'])
            if has_launch_block:
                for block in flow['blocks']:
                    if block.get('type') == LAUNCH_BLOCK_TYPE:
                        adapter_name = block.get('adapter')
                        workflow_id = block.get('workflowId')
                        url = f"https://app.kendra.io/{adapter_name}/{workflow_id}"
                        launch_block_urls.append(url)
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))

# Print URLs for flows containing launch blocks
print("\nURLs for Flows containing 'launch' blocks:")
for url in launch_block_urls:
    print(url)


Statistics:
Total 'launch' blocks before transformation: 100
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 88

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updategig"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updateartist"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "ada

In [42]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []
launch_block_urls = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def collect_launch_block_urls(blocks):
    """Collect URLs for all launch blocks in the blocks."""
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == LAUNCH_BLOCK_TYPE:
                adapter_name = block.get('adapter')
                workflow_id = block.get('workflowId')
                url = f"https://app.kendra.io/{adapter_name}/{workflow_id}"
                launch_block_urls.append(url)
            for key, value in block.items():
                if isinstance(value, list):
                    collect_launch_block_urls(value)
                elif isinstance(value, dict):
                    collect_launch_block_urls([value])

def process_blocks(blocks):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value)
                    elif isinstance(value, dict):
                        block[key] = process_nested_blocks(value)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    collect_launch_block_urls(blocks)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_nested_blocks(block):
    """Process nested blocks within a dictionary."""
    for key, value in block.items():
        if isinstance(value, list):
            block[key], _ = process_blocks(value)
        elif isinstance(value, dict):
            block[key] = process_nested_blocks(value)
    return block

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'])
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))

# Print URLs for flows containing launch blocks
print("\nURLs for Flows containing 'launch' blocks:")
for url in launch_block_urls:
    print(url)


Statistics:
Total 'launch' blocks before transformation: 100
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 88

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updategig"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updateartist"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "ada

In [43]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []
launch_block_urls = []

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def collect_launch_block_urls(blocks, flow):
    """Collect URLs for all launch blocks in the blocks."""
    launch_block_count = 0
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == LAUNCH_BLOCK_TYPE:
                adapter_name = block.get('adapter')
                workflow_id = block.get('workflowId')
                url = f"https://app.kendra.io/{adapter_name}/{workflow_id}"
                launch_block_urls.append((url, flow['title']))
                launch_block_count += 1
            for key, value in block.items():
                if isinstance(value, list):
                    launch_block_count += collect_launch_block_urls(value, flow)
                elif isinstance(value, dict):
                    launch_block_count += collect_launch_block_urls([value], flow)
    return launch_block_count

def process_blocks(blocks, flow):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value, flow)
                    elif isinstance(value, dict):
                        block[key] = process_nested_blocks(value, flow)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    launch_block_count = collect_launch_block_urls(blocks, flow)
    return updated_blocks, launch_block_count

def process_nested_blocks(block, flow):
    """Process nested blocks within a dictionary."""
    for key, value in block.items():
        if isinstance(value, list):
            block[key], _ = process_blocks(value, flow)
        elif isinstance(value, dict):
            block[key] = process_nested_blocks(value, flow)
    return block

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], launch_block_count = process_blocks(flow['blocks'], flow)
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            if launch_block_count > 0:
                print(f"Flow '{flow['title']}' contains {launch_block_count} launch blocks.")
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))

# Print URLs for flows containing launch blocks
print("\nURLs for Flows containing 'launch' blocks:")
unique_urls = set(launch_block_urls)
for url, title in unique_urls:
    print(f"{title}: {url}")


Flow 'Kendraio Backstage Add Gig' contains 2 launch blocks.
Flow 'Kendraio Backstage Add Gig' contains 2 launch blocks.
Flow 'Kendraio Backstage' contains 1 launch blocks.
Flow 'Kendraio Backstage Dashboard' contains 4 launch blocks.
Flow 'Kendraio Backstage Manage Artists' contains 2 launch blocks.
Flow 'Kendraio Backstage Manage Venues' contains 2 launch blocks.
Flow 'Kendraio Backstage' contains 1 launch blocks.
Flow 'Kendraio Backstage Update Artist' contains 2 launch blocks.
Flow 'Kendraio Backstage Update Gig' contains 1 launch blocks.
Flow 'Kendraio Backstage Update Venue' contains 2 launch blocks.
Flow 'Add Event' contains 1 launch blocks.
Flow 'Edit Event' contains 1 launch blocks.
Flow 'Edit Event' contains 1 launch blocks.
Flow 'Edit Event' contains 1 launch blocks.
Flow 'Edit Event' contains 1 launch blocks.
Flow 'List Events' contains 2 launch blocks.
Flow 'Bandsintown' contains 2 launch blocks.
Flow 'Contributions (Contributions)' contains 3 launch blocks.
Flow 'Edit Reco

In [44]:
import json
import requests
import copy

# Configuration
INPUT_FILE_URL = 'https://app.kendra.io/flows'
LAUNCH_BLOCK_TYPE = 'launch'
PARENT_BLOCK_TYPE = 'actions'
ALLOWED_IGNORE_KEYS = {'color', 'blockComment'}
FIXED_FLOWS_FILENAME = "fixed_flows.json"
UNFIXED_FLOWS_FILENAME = "unfixed_flows.json"

# Read JSON data from the URL
response = requests.get(INPUT_FILE_URL)
flows_data = response.json()

# Statistics and storage for unmodified blocks
stats = {
    'total_launch_blocks_before': 0,
    'total_launch_blocks_after': 0,
    'total_link_action_blocks': 0
}
unmodified_blocks = []
unfixed_flows = []
flow_launch_info = {}

def has_unexpected_properties(block, allowed_keys):
    """Check if the block contains unexpected properties."""
    return bool(set(block.keys()) - allowed_keys)

def get_value(data, keys, fallback=None):
    """Helper function to get direct or nested value with fallback."""
    if isinstance(keys, str):
        return data.get(keys, fallback)
    
    for key in keys:
        if isinstance(data, dict) and key in data:
            data = data[key]
        else:
            return fallback
    return data

def transform_launch_block(button, launch_block):
    """Transform a suitable launch block into a link-action block."""
    adapter_name = get_value(launch_block, 'adapter')
    workflow_id = get_value(launch_block, 'workflowId')
    adapter_name_getter = get_value(launch_block, ['valueGetters', 'adapter'])
    workflow_id_getter = get_value(launch_block, ['valueGetters', 'workflowId'])

    new_block = {
        "type": "link-action",
        "label": button.get('label', 'Open Flow'),
    }

    if adapter_name_getter:
        new_block["adapterNameGetter"] = adapter_name_getter
    else:
        new_block["adapterName"] = adapter_name

    if workflow_id_getter:
        new_block["workflowIdGetter"] = workflow_id_getter
    else:
        new_block["workflowId"] = workflow_id

    stats['total_link_action_blocks'] += 1
    return new_block

def can_transform_buttons(buttons):
    """Check if all buttons in the actions block can be transformed."""
    for button in buttons:
        if 'blocks' not in button or len(button['blocks']) != 1:
            return False
        launch_block = button['blocks'][0]
        if launch_block.get('type') != LAUNCH_BLOCK_TYPE:
            return False
        if has_unexpected_properties(launch_block, {'type', 'adapter', 'workflowId', 'valueGetters'}):
            return False
        if has_unexpected_properties(button, {'label', 'color', 'blocks', 'enabled'}):
            return False
    return True

def process_buttons(buttons):
    """Process and transform buttons."""
    transformed_buttons = []
    for button in buttons:
        launch_block = button['blocks'][0]
        new_block = transform_launch_block(button, launch_block)
        transformed_buttons.append(new_block)
    return transformed_buttons

def collect_launch_block_info(blocks, flow_id):
    """Collect info for all launch blocks in the blocks."""
    count = 0
    urls = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == LAUNCH_BLOCK_TYPE:
                adapter_name = block.get('adapter')
                workflow_id = block.get('workflowId')
                url = f"https://app.kendra.io/{adapter_name}/{workflow_id}"
                urls.append(url)
                count += 1
            for key, value in block.items():
                if isinstance(value, list):
                    sub_count, sub_urls = collect_launch_block_info(value, flow_id)
                    count += sub_count
                    urls.extend(sub_urls)
                elif isinstance(value, dict):
                    sub_count, sub_urls = collect_launch_block_info([value], flow_id)
                    count += sub_count
                    urls.extend(sub_urls)
    if count > 0:
        if flow_id not in flow_launch_info:
            flow_launch_info[flow_id] = {'count': 0, 'urls': []}
        flow_launch_info[flow_id]['count'] += count
        flow_launch_info[flow_id]['urls'].extend(urls)
    return count, urls

def process_blocks(blocks, flow_id):
    """Recursively process and transform blocks."""
    updated_blocks = []
    for block in blocks:
        if isinstance(block, dict):
            if block.get('type') == PARENT_BLOCK_TYPE:
                buttons = block.get('buttons', [])
                if can_transform_buttons(buttons):
                    transformed_buttons = process_buttons(buttons)
                    updated_blocks.extend(transformed_buttons)
                    stats['total_launch_blocks_before'] += len(buttons)
                else:
                    updated_blocks.append(block)
                    for button in buttons:
                        unmodified_blocks.append({
                            "block": button,
                            "reason": "Button does not meet transformation criteria"
                        })
            else:
                for key, value in block.items():
                    if isinstance(value, list):
                        block[key], _ = process_blocks(value, flow_id)
                    elif isinstance(value, dict):
                        block[key] = process_nested_blocks(value, flow_id)
                updated_blocks.append(block)
        else:
            updated_blocks.append(block)
    collect_launch_block_info(blocks, flow_id)
    return updated_blocks, any(isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE for block in blocks)

def process_nested_blocks(block, flow_id):
    """Process nested blocks within a dictionary."""
    for key, value in block.items():
        if isinstance(value, list):
            block[key], _ = process_blocks(value, flow_id)
        elif isinstance(value, dict):
            block[key] = process_nested_blocks(value, flow_id)
    return block

def process_flows(flows):
    """Process each flow and store in fixed_flows and unfixed_flows."""
    fixed_flows = []
    for flow in flows:
        original_flow = copy.deepcopy(flow)
        flow_id = flow.get('id', 'unknown')
        if 'blocks' in flow:
            stats['total_launch_blocks_before'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
            flow['blocks'], _ = process_blocks(flow['blocks'], flow_id)
            stats['total_launch_blocks_after'] += sum(1 for block in flow['blocks'] if isinstance(block, dict) and block.get('type') == LAUNCH_BLOCK_TYPE)
        fixed_flows.append(flow)
        unfixed_flows.append(original_flow)  # Include all original flows for diff
    return fixed_flows

# Process the flows
fixed_flows = process_flows(flows_data)

# Write the fixed flows to a JSON file
with open(FIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(fixed_flows, outfile, indent=2)

# Write the unfixed flows to a JSON file
with open(UNFIXED_FLOWS_FILENAME, "w") as outfile:
    json.dump(unfixed_flows, outfile, indent=2)

# Print statistics and unmodified blocks
print("Statistics:")
print(f"Total 'launch' blocks before transformation: {stats['total_launch_blocks_before']}")
print(f"Total 'launch' blocks after transformation: {stats['total_launch_blocks_after']}")
print(f"Total 'link-action' blocks created: {stats['total_link_action_blocks']}")
print("\nUnmodified Launch Blocks:")
for entry in unmodified_blocks:
    print("Reason:", entry["reason"])
    print(json.dumps(entry["block"], indent=2))

# Print URLs for flows containing launch blocks and count
print("\nFlow Launch Block Info:")
for flow_id, info in flow_launch_info.items():
    print(f"Flow ID: {flow_id}")
    print(f"Count of 'launch' blocks: {info['count']}")
    print("URLs:")
    for url in info['urls']:
        print(url)
    print()


Statistics:
Total 'launch' blocks before transformation: 100
Total 'launch' blocks after transformation: 12
Total 'link-action' blocks created: 88

Unmodified Launch Blocks:
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updategig"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "adapter": "backstage",
      "workflowId": "updateartist"
    }
  ],
  "enabled": true
}
Reason: Button does not meet transformation criteria
{
  "label": "Edit",
  "color": "primary",
  "blocks": [
    {
      "type": "variable-set",
      "name": "flow_data"
    },
    {
      "type": "launch",
      "ada